# Class 6 — Data Validation & Profiling

## NYC Taxi Operations Intelligence Pipeline

This notebook profiles and validates the July 2026 reporting dataset
retrieved during Class 5.

The objective is to identify meaningful data-quality issues, establish
business-oriented validation rules, document assumptions and
limitations, and determine whether the data is suitable for producing
the defined operational metrics.

No data is silently modified during the profiling stage.

In [1]:
import pandas as pd

trip_path = "../data/raw/trips/yellow_tripdata_2026-07.parquet"
zone_path = "../data/raw/zones/taxi_zone_lookup.csv"

trips = pd.read_parquet(trip_path)
zones = pd.read_csv(zone_path)

july_2026_trips = trips[
    (trips["tpep_pickup_datetime"] >= "2026-07-01") &
    (trips["tpep_pickup_datetime"] < "2026-08-01")
].copy()

print("July 2026 rows:", len(july_2026_trips))

July 2026 rows: 3530063


## 6A.1 — Basic Dataset Profile

This section establishes the basic structure of the July 2026
reporting dataset before applying any validation or cleaning rules.

The profiling stage is observational. No records are modified.

In [2]:
print("========== BASIC DATASET PROFILE ==========")

print("Rows:", len(july_2026_trips))
print("Columns:", len(july_2026_trips.columns))

print("\nColumn names:")
print(july_2026_trips.columns.tolist())

print("\nData types:")
print(july_2026_trips.dtypes)

print("============================================")

========== BASIC DATASET PROFILE ==========
Rows: 3530063
Columns: 21

Column names:
['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee', 'cbd_congestion_fee', 'request_source']

Data types:
VendorID                          int32
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count                 float64
trip_distance                   float64
RatecodeID                      float64
store_and_fwd_flag                  str
PULocationID                      int32
DOLocationID                      int32
payment_type                      int64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount     

## 6A.2 — Missing Value Profile

Missing values are measured for each column to identify fields that
may affect trip metrics, joins, or validation rules.

In [3]:
missing_count = july_2026_trips.isna().sum()

missing_percentage = (
    july_2026_trips.isna().mean() * 100
)

missing_summary = pd.DataFrame({
    "missing_count": missing_count,
    "missing_percentage": missing_percentage
}).sort_values(
    "missing_count",
    ascending=False
)

print(missing_summary)

                       missing_count  missing_percentage
request_source               2560646           72.538252
store_and_fwd_flag            969727           27.470530
congestion_surcharge          969727           27.470530
RatecodeID                    969727           27.470530
passenger_count               969727           27.470530
Airport_fee                   969727           27.470530
tpep_dropoff_datetime              0            0.000000
tpep_pickup_datetime               0            0.000000
VendorID                           0            0.000000
trip_distance                      0            0.000000
payment_type                       0            0.000000
PULocationID                       0            0.000000
DOLocationID                       0            0.000000
mta_tax                            0            0.000000
extra                              0            0.000000
fare_amount                        0            0.000000
tip_amount                     

In [4]:
kpi_columns = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "trip_distance"
]

kpi_missing_profile = pd.DataFrame({
    "missing_count": july_2026_trips[kpi_columns].isna().sum(),
    "missing_percentage": (
        july_2026_trips[kpi_columns].isna().mean() * 100
    )
})

print(kpi_missing_profile)

                       missing_count  missing_percentage
tpep_pickup_datetime               0                 0.0
tpep_dropoff_datetime              0                 0.0
PULocationID                       0                 0.0
DOLocationID                       0                 0.0
trip_distance                      0                 0.0


## 6A.4 — Duplicate Profile

Duplicate records are measured before any transformation or removal.

An exact duplicate and a duplicate based on selected trip fields are
examined separately because identical trip attributes do not
necessarily prove that two records represent the same real-world trip.

In [5]:
exact_duplicates = july_2026_trips.duplicated().sum()

print("Exact duplicate rows:", exact_duplicates)

Exact duplicate rows: 0


In [6]:
trip_key_columns = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "trip_distance"
]

core_duplicates = july_2026_trips.duplicated(
    subset=trip_key_columns
).sum()

print(
    "Rows duplicated across core trip fields:",
    core_duplicates
)

Rows duplicated across core trip fields: 71485


## 6A.5 — Trip Distance Profile

Trip distance is profiled because it is used in the Average Trip
Distance KPI.

The distribution is inspected before defining any validity rule.

In [7]:
distance_profile = july_2026_trips["trip_distance"].describe()

print(distance_profile)

count    3.530063e+06
mean     5.547570e+00
std      5.762057e+02
min      0.000000e+00
25%      1.030000e+00
50%      1.900000e+00
75%      3.940000e+00
max      3.181291e+05
Name: trip_distance, dtype: float64


In [8]:
print(
    "Zero-distance trips:",
    (july_2026_trips["trip_distance"] == 0).sum()
)

print(
    "Negative-distance trips:",
    (july_2026_trips["trip_distance"] < 0).sum()
)

Zero-distance trips: 128322
Negative-distance trips: 0


In [9]:
print(
    "Missing-distance trips:",
    july_2026_trips["trip_distance"].isna().sum()
)

Missing-distance trips: 0


## 6A.6 — Timestamp Profile

Pickup and dropoff timestamps are profiled because trip duration is
one of the primary business KPIs.

The derived duration is measured in minutes.

No duration-based records are removed at this stage.

In [12]:
pickup = july_2026_trips["tpep_pickup_datetime"]
dropoff = july_2026_trips["tpep_dropoff_datetime"]

print("Pickup minimum:", pickup.min())
print("Pickup maximum:", pickup.max())

print("Dropoff minimum:", dropoff.min())
print("Dropoff maximum:", dropoff.max())

Pickup minimum: 2026-07-01 00:00:00
Pickup maximum: 2026-07-31 23:59:59
Dropoff minimum: 2026-07-01 00:00:15
Dropoff maximum: 2026-08-01 21:49:34


In [13]:
trip_duration_minutes = (
    dropoff - pickup
).dt.total_seconds() / 60

print(trip_duration_minutes.describe())

count    3.530063e+06
mean     1.729142e+01
std      2.707407e+01
min     -1.666667e-01
25%      8.483333e+00
50%      1.416667e+01
75%      2.216667e+01
max      1.716542e+04
dtype: float64


In [14]:
print(
    "Missing duration:",
    trip_duration_minutes.isna().sum()
)

print(
    "Zero-duration trips:",
    (trip_duration_minutes == 0).sum()
)

print(
    "Negative-duration trips:",
    (trip_duration_minutes < 0).sum()
)

Missing duration: 0
Zero-duration trips: 42315
Negative-duration trips: 1


In [15]:
print("Shortest durations:")
print(
    trip_duration_minutes
    .sort_values()
    .head(20)
)

print("\nLongest durations:")
print(
    trip_duration_minutes
    .sort_values(ascending=False)
    .head(20)
)

Shortest durations:
3344248   -0.166667
1644927    0.000000
80838      0.000000
1285822    0.000000
430683     0.000000
1783744    0.000000
640477     0.000000
640517     0.000000
1744894    0.000000
1610286    0.000000
1610267    0.000000
1610266    0.000000
2276956    0.000000
589652     0.000000
2470680    0.000000
818614     0.000000
818630     0.000000
1404792    0.000000
1893727    0.000000
1893726    0.000000
dtype: float64

Longest durations:
3129446    17165.416667
1475252    17165.416667
2628970     7170.616667
372547      5678.683333
1351933     5347.400000
1357364     5260.183333
1685941     4357.966667
428139      4253.850000
410820      4227.116667
686547      4173.866667
2196545     3821.483333
1358218     3786.766667
510027      3285.216667
1394987     3001.000000
3081312     3001.000000
2388824     2892.183333
313175      2781.433333
1779681     2752.116667
495054      2704.950000
1392597     2583.450000
dtype: float64


In [16]:
location_columns = [
    "PULocationID",
    "DOLocationID"
]

print(
    july_2026_trips[location_columns].describe()
)

       PULocationID  DOLocationID
count  3.530063e+06  3.530063e+06
mean   1.610132e+02  1.602974e+02
std    6.638627e+01  7.046886e+01
min    1.000000e+00  1.000000e+00
25%    1.140000e+02  1.070000e+02
50%    1.610000e+02  1.620000e+02
75%    2.310000e+02  2.330000e+02
max    2.650000e+02  2.650000e+02


In [17]:
print("Missing pickup locations:")
print(july_2026_trips["PULocationID"].isna().sum())

print("Missing dropoff locations:")
print(july_2026_trips["DOLocationID"].isna().sum())

Missing pickup locations:
0
Missing dropoff locations:
0


In [18]:
print(
    "Unique pickup locations:",
    july_2026_trips["PULocationID"].nunique()
)

print(
    "Unique dropoff locations:",
    july_2026_trips["DOLocationID"].nunique()
)

Unique pickup locations: 260
Unique dropoff locations: 261


## 6B.1 — Investigate Missing Values

Missing values are investigated based on whether the affected field
is required for a business metric or relationship.

A missing value is not automatically treated as an invalid record.
Its impact depends on the role of the field in the analysis.

In [19]:
critical_columns = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "trip_distance"
]

missing_in_critical = pd.DataFrame({
    "missing_count": july_2026_trips[critical_columns].isna().sum(),
    "missing_percentage": (
        july_2026_trips[critical_columns].isna().mean() * 100
    )
})

print(missing_in_critical)

                       missing_count  missing_percentage
tpep_pickup_datetime               0                 0.0
tpep_dropoff_datetime              0                 0.0
PULocationID                       0                 0.0
DOLocationID                       0                 0.0
trip_distance                      0                 0.0


In [20]:
duplicate_mask = july_2026_trips.duplicated(
    subset=trip_key_columns,
    keep=False
)

duplicate_examples = july_2026_trips[
    duplicate_mask
].sort_values(trip_key_columns)

print("Rows involved in core-field duplication:", len(duplicate_examples))

print(
    duplicate_examples[
        trip_key_columns
    ].head(20)
)

Rows involved in core-field duplication: 142970
     tpep_pickup_datetime tpep_dropoff_datetime  PULocationID  DOLocationID  \
245   2026-07-01 00:00:07   2026-07-01 00:09:49           237           226   
246   2026-07-01 00:00:07   2026-07-01 00:09:49           237           226   
1229  2026-07-01 00:01:47   2026-07-01 00:22:50           161            49   
1230  2026-07-01 00:01:47   2026-07-01 00:22:50           161            49   
434   2026-07-01 00:08:35   2026-07-01 00:24:37           209           190   
435   2026-07-01 00:08:35   2026-07-01 00:24:37           209           190   
392   2026-07-01 00:13:43   2026-07-01 00:15:59           138           138   
393   2026-07-01 00:13:43   2026-07-01 00:15:59           138           138   
1376  2026-07-01 00:22:26   2026-07-01 01:10:32           198           174   
1377  2026-07-01 00:22:26   2026-07-01 01:10:32           198           174   
551   2026-07-01 00:23:39   2026-07-01 00:23:49            79            79   
552 

## 6B.3 — Investigate Trip Duration

Trip duration is derived from pickup and dropoff timestamps:

duration = dropoff timestamp - pickup timestamp

The investigation looks for missing, zero, negative, and extreme
durations before defining the validation rule.

In [21]:
july_2026_trips["trip_duration_minutes"] = (
    july_2026_trips["tpep_dropoff_datetime"]
    - july_2026_trips["tpep_pickup_datetime"]
).dt.total_seconds() / 60

In [23]:
duration_quality = {
    "missing": july_2026_trips["trip_duration_minutes"].isna().sum(),
    "zero": (
        july_2026_trips["trip_duration_minutes"] == 0
    ).sum(),
    "negative": (
        july_2026_trips["trip_duration_minutes"] < 0
    ).sum()
}

print(duration_quality)

{'missing': np.int64(0), 'zero': np.int64(42315), 'negative': np.int64(1)}


In [24]:
duration_quality = {
    "missing": july_2026_trips["trip_duration_minutes"].isna().sum(),
    "zero": (
        july_2026_trips["trip_duration_minutes"] == 0
    ).sum(),
    "negative": (
        july_2026_trips["trip_duration_minutes"] < 0
    ).sum()
}

print(duration_quality)

{'missing': np.int64(0), 'zero': np.int64(42315), 'negative': np.int64(1)}


In [25]:
duration_percentiles = (
    july_2026_trips["trip_duration_minutes"]
    .quantile([
        0.01,
        0.05,
        0.25,
        0.50,
        0.75,
        0.95,
        0.99,
        0.999
    ])
)

print(duration_percentiles)

0.010     0.000000
0.050     3.316667
0.250     8.483333
0.500    14.166667
0.750    22.166667
0.950    40.716667
0.990    62.616667
0.999    93.850000
Name: trip_duration_minutes, dtype: float64


In [26]:
longest_trips = (
    july_2026_trips
    .sort_values(
        "trip_duration_minutes",
        ascending=False
    )
    [
        [
            "tpep_pickup_datetime",
            "tpep_dropoff_datetime",
            "trip_duration_minutes",
            "trip_distance",
            "PULocationID",
            "DOLocationID"
        ]
    ]
    .head(20)
)

print(longest_trips)

        tpep_pickup_datetime tpep_dropoff_datetime  trip_duration_minutes  \
3129446  2026-07-19 14:32:52   2026-07-31 12:38:17           17165.416667   
1475252  2026-07-19 14:32:52   2026-07-31 12:38:17           17165.416667   
2628970  2026-07-03 13:09:53   2026-07-08 12:40:30            7170.616667   
372547   2026-07-06 15:09:36   2026-07-10 13:48:17            5678.683333   
1351933  2026-07-17 22:54:33   2026-07-21 16:01:57            5347.400000   
1357364  2026-07-17 23:26:09   2026-07-21 15:06:20            5260.183333   
1685941  2026-07-22 05:30:01   2026-07-25 06:07:59            4357.966667   
428139   2026-07-07 10:36:40   2026-07-10 09:30:31            4253.850000   
410820   2026-07-06 22:05:14   2026-07-09 20:32:21            4227.116667   
686547   2026-07-10 10:56:29   2026-07-13 08:30:21            4173.866667   
2196545  2026-07-28 00:52:49   2026-07-30 16:34:18            3821.483333   
1358218  2026-07-17 23:30:12   2026-07-20 14:36:58            3786.766667   

## 6B.4 — Investigate Trip Distance

Trip distance is investigated because it directly affects the
Average Trip Distance KPI.

The investigation examines missing, zero, negative, and extreme
values before defining a business validation rule.

In [27]:
distance_quality = {
    "missing": july_2026_trips["trip_distance"].isna().sum(),
    "zero": (
        july_2026_trips["trip_distance"] == 0
    ).sum(),
    "negative": (
        july_2026_trips["trip_distance"] < 0
    ).sum()
}

print(distance_quality)

{'missing': np.int64(0), 'zero': np.int64(128322), 'negative': np.int64(0)}


In [28]:
distance_percentiles = (
    july_2026_trips["trip_distance"]
    .quantile([
        0.01,
        0.05,
        0.25,
        0.50,
        0.75,
        0.95,
        0.99,
        0.999
    ])
)

print(distance_percentiles)

0.010     0.00000
0.050     0.25000
0.250     1.03000
0.500     1.90000
0.750     3.94000
0.950    12.05000
0.990    19.51000
0.999    33.92938
Name: trip_distance, dtype: float64


In [29]:
largest_distances = (
    july_2026_trips
    .sort_values(
        "trip_distance",
        ascending=False
    )
    [
        [
            "trip_distance",
            "trip_duration_minutes",
            "PULocationID",
            "DOLocationID"
        ]
    ]
    .head(20)
)

print(largest_distances)

         trip_distance  trip_duration_minutes  PULocationID  DOLocationID
3458861      318129.10                   31.0           148            28
3208119      317017.94                   24.0            87           143
3414440      270530.29                   33.0            66            68
3209680      247657.16                   16.0           163            90
2637969      217780.00                   13.0           137           100
3273511      189585.48                   12.0           107           162
2917549      188214.94                   20.0           151           119
2715885      188030.58                    8.0            24           142
2987590      181291.03                   16.0            13           140
3361769      178321.52                    8.0           142           246
3295556      177970.82                    7.0           140           140
2953708      175950.76                    6.0            41           151
2841814      173272.88                

In [30]:
zone_ids = set(
    zones["LocationID"].dropna()
)

pickup_ids = set(
    july_2026_trips["PULocationID"].dropna()
)

dropoff_ids = set(
    july_2026_trips["DOLocationID"].dropna()
)

unmatched_pickup_ids = pickup_ids - zone_ids
unmatched_dropoff_ids = dropoff_ids - zone_ids

print(
    "Unmatched pickup IDs:",
    len(unmatched_pickup_ids)
)

print(
    "Unmatched dropoff IDs:",
    len(unmatched_dropoff_ids)
)

Unmatched pickup IDs: 0
Unmatched dropoff IDs: 0


In [31]:
print(
    "Missing pickup IDs:",
    july_2026_trips["PULocationID"].isna().sum()
)

print(
    "Missing dropoff IDs:",
    july_2026_trips["DOLocationID"].isna().sum()
)

Missing pickup IDs: 0
Missing dropoff IDs: 0


In [32]:
print(
    july_2026_trips[
        [
            "trip_duration_minutes",
            "trip_distance"
        ]
    ].corr()
)

                       trip_duration_minutes  trip_distance
trip_duration_minutes               1.000000       0.004791
trip_distance                       0.004791       1.000000


In [33]:
zero_distance_positive_duration = july_2026_trips[
    (july_2026_trips["trip_distance"] == 0) &
    (july_2026_trips["trip_duration_minutes"] > 0)
]

print(
    "Zero-distance trips with positive duration:",
    len(zero_distance_positive_duration)
)

Zero-distance trips with positive duration: 127071


In [34]:
print(
    zero_distance_positive_duration[
        [
            "tpep_pickup_datetime",
            "tpep_dropoff_datetime",
            "trip_duration_minutes",
            "trip_distance",
            "PULocationID",
            "DOLocationID"
        ]
    ].head(20)
)

     tpep_pickup_datetime tpep_dropoff_datetime  trip_duration_minutes  \
36    2026-07-01 00:20:00   2026-07-01 00:20:12               0.200000   
86    2026-07-01 00:32:49   2026-07-01 00:32:54               0.083333   
333   2026-07-01 00:02:54   2026-07-01 00:03:03               0.150000   
392   2026-07-01 00:13:43   2026-07-01 00:15:59               2.266667   
393   2026-07-01 00:13:43   2026-07-01 00:15:59               2.266667   
440   2026-07-01 00:49:07   2026-07-01 00:49:11               0.066667   
551   2026-07-01 00:23:39   2026-07-01 00:23:49               0.166667   
552   2026-07-01 00:23:39   2026-07-01 00:23:49               0.166667   
783   2026-07-01 00:31:16   2026-07-01 00:31:20               0.066667   
786   2026-07-01 00:32:38   2026-07-01 00:32:42               0.066667   
791   2026-07-01 00:32:07   2026-07-01 00:32:10               0.050000   
816   2026-07-01 00:22:56   2026-07-01 00:23:02               0.100000   
837   2026-07-01 00:22:36   2026-07-01

## 6B.7 — Data Quality Findings

The profiling results are converted into explicit findings before
validation rules are applied.

In [36]:
quality_findings = pd.DataFrame([
    {
        "issue": "Missing critical fields",
        "field": "Pickup/dropoff/location/distance fields",
        "observed": "See profiling results",
        "business_impact": "May prevent calculation of specific metrics",
        "status": "Investigated"
    },
    {
        "issue": "Duplicate trip records",
        "field": "Core trip fields",
        "observed": "See duplicate analysis",
        "business_impact": "Could inflate trip volume if true duplicates exist",
        "status": "Investigated"
    },
    {
        "issue": "Invalid trip duration",
        "field": "Derived duration",
        "observed": "See duration analysis",
        "business_impact": "Can distort average and median duration",
        "status": "Investigated"
    },
    {
        "issue": "Invalid trip distance",
        "field": "trip_distance",
        "observed": "See distance analysis",
        "business_impact": "Can distort average trip distance",
        "status": "Investigated"
    },
    {
        "issue": "Location reference integrity",
        "field": "PULocationID / DOLocationID",
        "observed": "See reference validation",
        "business_impact": "Can prevent geographic analysis",
        "status": "Investigated"
    }
])

print(quality_findings.to_string(index=False))

                       issue                                   field                 observed                                    business_impact       status
     Missing critical fields Pickup/dropoff/location/distance fields    See profiling results        May prevent calculation of specific metrics Investigated
      Duplicate trip records                        Core trip fields   See duplicate analysis Could inflate trip volume if true duplicates exist Investigated
       Invalid trip duration                        Derived duration    See duration analysis            Can distort average and median duration Investigated
       Invalid trip distance                           trip_distance    See distance analysis                  Can distort average trip distance Investigated
Location reference integrity             PULocationID / DOLocationID See reference validation                    Can prevent geographic analysis Investigated


## 6C — Define Validation Rules

Validation rules translate the observed data-quality findings into
explicit checks that can be executed consistently.

The rules are based on:
1. Required fields for the defined metrics
2. Logical consistency between fields
3. Referential integrity with the taxi zone lookup
4. Business interpretation of the trip record

The validation stage does not silently modify the raw data.
Records are classified according to the validation results.

                    JULY TRIP
                       │
          ┌────────────┼────────────┐
          ▼            ▼            ▼
      Completeness  Validity    Integrity
          │            │            │
          ▼            ▼            ▼
       Missing     Duration      Location
       fields      Distance      reference
          │            │            │
          └────────────┼────────────┘
                       ▼
                Validation Result

In [37]:
required_timestamp_rule = (
    july_2026_trips["tpep_pickup_datetime"].notna()
    &
    july_2026_trips["tpep_dropoff_datetime"].notna()
)

print("Trips passing timestamp rule:", required_timestamp_rule.sum())
print("Trips failing timestamp rule:", (~required_timestamp_rule).sum())

Trips passing timestamp rule: 3530063
Trips failing timestamp rule: 0


In [38]:
chronology_rule = (
    july_2026_trips["tpep_dropoff_datetime"]
    >=
    july_2026_trips["tpep_pickup_datetime"]
)

print("Trips passing chronology rule:", chronology_rule.sum())
print("Trips failing chronology rule:", (~chronology_rule).sum())

Trips passing chronology rule: 3530062
Trips failing chronology rule: 1


In [39]:
location_completeness_rule = (
    july_2026_trips["PULocationID"].notna()
    &
    july_2026_trips["DOLocationID"].notna()
)

print(
    "Trips passing location completeness:",
    location_completeness_rule.sum()
)

print(
    "Trips failing location completeness:",
    (~location_completeness_rule).sum()
)

Trips passing location completeness: 3530063
Trips failing location completeness: 0


In [40]:
zone_ids = set(
    zones["LocationID"].dropna()
)

pickup_reference_rule = july_2026_trips["PULocationID"].isin(
    zone_ids
)

dropoff_reference_rule = july_2026_trips["DOLocationID"].isin(
    zone_ids
)

location_reference_rule = (
    pickup_reference_rule &
    dropoff_reference_rule
)

print(
    "Trips passing location reference rule:",
    location_reference_rule.sum()
)

print(
    "Trips failing location reference rule:",
    (~location_reference_rule).sum()
)

Trips passing location reference rule: 3530063
Trips failing location reference rule: 0


In [41]:
distance_rule = (
    july_2026_trips["trip_distance"].notna()
    &
    (july_2026_trips["trip_distance"] >= 0)
)

print(
    "Trips passing distance rule:",
    distance_rule.sum()
)

print(
    "Trips failing distance rule:",
    (~distance_rule).sum()
)

Trips passing distance rule: 3530063
Trips failing distance rule: 0


In [42]:
required_fields_rule = (
    july_2026_trips["tpep_pickup_datetime"].notna()
    &
    july_2026_trips["tpep_dropoff_datetime"].notna()
    &
    july_2026_trips["PULocationID"].notna()
    &
    july_2026_trips["DOLocationID"].notna()
    &
    july_2026_trips["trip_distance"].notna()
)

print(
    "Trips passing required-field rule:",
    required_fields_rule.sum()
)

print(
    "Trips failing required-field rule:",
    (~required_fields_rule).sum()
)

Trips passing required-field rule: 3530063
Trips failing required-field rule: 0


In [43]:
validation_pass = (
    required_fields_rule
    &
    chronology_rule
    &
    location_reference_rule
    &
    distance_rule
)

In [44]:
print("Total July trips:", len(july_2026_trips))

print(
    "Trips passing all core validation rules:",
    validation_pass.sum()
)

print(
    "Trips failing at least one core validation rule:",
    (~validation_pass).sum()
)

Total July trips: 3530063
Trips passing all core validation rules: 3530062
Trips failing at least one core validation rule: 1


In [45]:
validation_results = pd.DataFrame({
    "missing_required_field": ~required_fields_rule,
    "invalid_chronology": ~chronology_rule,
    "invalid_location_reference": ~location_reference_rule,
    "invalid_distance": ~distance_rule
})

print(
    validation_results.sum().sort_values(
        ascending=False
    )
)

invalid_chronology            1
missing_required_field        0
invalid_location_reference    0
invalid_distance              0
dtype: int64


In [46]:
duration_denominator = (
    july_2026_trips[
        [
            "tpep_pickup_datetime",
            "tpep_dropoff_datetime"
        ]
    ].notna().all(axis=1)
).sum()

invalid_duration_count = (
    (
        july_2026_trips["tpep_dropoff_datetime"]
        <
        july_2026_trips["tpep_pickup_datetime"]
    )
    &
    july_2026_trips[
        [
            "tpep_pickup_datetime",
            "tpep_dropoff_datetime"
        ]
    ].notna().all(axis=1)
).sum()

invalid_duration_rate = (
    invalid_duration_count
    / duration_denominator
    * 100
)

print("Invalid duration count:", invalid_duration_count)
print("Duration denominator:", duration_denominator)
print("Invalid Trip Duration Rate:", invalid_duration_rate)

Invalid duration count: 1
Duration denominator: 3530063
Invalid Trip Duration Rate: 2.832810632558116e-05


In [47]:
location_available = (
    july_2026_trips["PULocationID"].notna()
    &
    july_2026_trips["DOLocationID"].notna()
)

location_valid = (
    july_2026_trips["PULocationID"].isin(zone_ids)
    &
    july_2026_trips["DOLocationID"].isin(zone_ids)
)

invalid_location_count = (
    location_available
    & ~location_valid
).sum()

In [48]:
location_issue = ~location_valid

location_denominator = len(july_2026_trips)

invalid_location_count = location_issue.sum()

invalid_location_rate = (
    invalid_location_count
    / location_denominator
    * 100
)

print("Location issue count:", invalid_location_count)
print("Location denominator:", location_denominator)
print("Invalid/Unmatched Location Rate:", invalid_location_rate)

Location issue count: 0
Location denominator: 3530063
Invalid/Unmatched Location Rate: 0.0


## 6C.13 — Validation Rule Catalogue

| Rule | Definition | Business Reason | Treatment |
|---|---|---|---|
| Required timestamps | Pickup and dropoff timestamps must exist | Required for trip duration | Flag |
| Chronology | Dropoff >= pickup | Negative duration is logically impossible | Flag |
| Location completeness | Pickup and dropoff IDs must exist | Required for geographic analysis | Flag |
| Location reference | Location IDs must exist in zone lookup | Required for zone-level analysis | Flag |
| Distance completeness | Trip distance must exist | Required for distance KPI | Flag |
| Distance validity | Trip distance must be >= 0 | Negative distance is logically invalid | Flag |
| Extreme duration | Long duration is reported but not automatically invalid | Long trips may be legitimate | Investigate |
| Extreme distance | Large distance is reported but not automatically invalid | Long trips may be legitimate | Investigate |

### Validation Philosophy

The validation rules distinguish between:

1. Logically invalid records
2. Incomplete records
3. Referential-integrity issues
4. Unusual but potentially legitimate records

Records are not silently deleted or corrected during profiling and
validation.

Extreme values are investigated separately from hard validity rules
when the available evidence does not establish that they are invalid.

This preserves the distinction between data quality problems and
legitimate operational variation.


## 6D.1 — Investigate Invalid Chronology Record

One trip failed the timestamp chronology rule because its dropoff
timestamp occurs before its pickup timestamp.

The record is inspected before deciding how it should be treated
in downstream metrics.

In [49]:
invalid_chronology_records = july_2026_trips[
    july_2026_trips["tpep_dropoff_datetime"]
    <
    july_2026_trips["tpep_pickup_datetime"]
].copy()

print("Invalid chronology records:", len(invalid_chronology_records))

print(
    invalid_chronology_records[
        [
            "tpep_pickup_datetime",
            "tpep_dropoff_datetime",
            "PULocationID",
            "DOLocationID",
            "trip_distance"
        ]
    ]
)

Invalid chronology records: 1
        tpep_pickup_datetime tpep_dropoff_datetime  PULocationID  \
3344248  2026-07-25 22:24:27   2026-07-25 22:24:17           148   

         DOLocationID  trip_distance  
3344248           148           3.65  


In [50]:
invalid_chronology_records["trip_duration_minutes"] = (
    invalid_chronology_records["tpep_dropoff_datetime"]
    -
    invalid_chronology_records["tpep_pickup_datetime"]
).dt.total_seconds() / 60

print(
    invalid_chronology_records[
        [
            "tpep_pickup_datetime",
            "tpep_dropoff_datetime",
            "trip_duration_minutes",
            "trip_distance",
            "PULocationID",
            "DOLocationID"
        ]
    ]
)

        tpep_pickup_datetime tpep_dropoff_datetime  trip_duration_minutes  \
3344248  2026-07-25 22:24:27   2026-07-25 22:24:17              -0.166667   

         trip_distance  PULocationID  DOLocationID  
3344248           3.65           148           148  


In [51]:
location_issue_count = (
    (~july_2026_trips["PULocationID"].isin(zone_ids))
    |
    (~july_2026_trips["DOLocationID"].isin(zone_ids))
).sum()

location_issue_rate = (
    location_issue_count
    / len(july_2026_trips)
    * 100
)

print("Location issue count:", location_issue_count)
print("Location issue rate:", location_issue_rate)

Location issue count: 0
Location issue rate: 0.0


## 6D.2 — Validation Results

The July 2026 reporting dataset contains 3,530,063 trip records.

The validation checks found:

- No missing pickup or dropoff timestamps.
- No missing pickup or dropoff location IDs.
- No unmatched pickup or dropoff location IDs.
- No negative or missing trip distances.
- One trip with invalid timestamp chronology.
- 3,530,062 trips passed all core validation rules.

The invalid chronology record is retained in the raw data but should
not contribute to duration-based metrics because its derived duration
is logically invalid.

No broad filtering or silent correction is applied to the raw source.

## 6D.3 — Final Validation Rule Catalogue

| Rule | Definition | Result | Status |
|---|---|---:|---|
| Timestamp completeness | Pickup and dropoff timestamps must exist | 0 failures | PASS |
| Timestamp chronology | Dropoff >= pickup | 1 failure | REVIEW |
| Location completeness | Pickup and dropoff IDs must exist | 0 failures | PASS |
| Location reference | IDs must exist in zone lookup | 0 failures | PASS |
| Distance completeness | Distance must exist | 0 failures | PASS |
| Distance validity | Distance >= 0 | 0 failures | PASS |
| Core validation | All core rules must pass | 1 failure | REVIEW |

## 6D.4 — Assumptions and Limitations

### Assumptions

1. The reporting period is defined by taxi pickup timestamp.
2. A trip with dropoff timestamp earlier than pickup timestamp is
   considered invalid for duration-based analysis.
3. A LocationID is considered valid when it exists in the retrieved
   taxi zone reference.
4. Negative trip distance is considered invalid.
5. Zero trip distance is not automatically classified as invalid
   because the available data alone does not establish that every
   zero-distance record is erroneous.

### Limitations

1. The validation rules identify logical and referential data-quality
   issues but cannot establish whether every unusual operational
   value is factually incorrect.
2. Long trip durations and large trip distances are treated as
   potentially legitimate observations unless a defensible business
   rule establishes otherwise.
3. The taxi zone lookup validates LocationID references but does not
   establish whether the geographic assignment is operationally
   correct.
4. The analysis uses the selected July 2026 reporting period and
   should not automatically be generalized to other months.